# ⚖️ EVALUASI PERBANDINGAN — XGBoost vs LSTM
## Tesis: Pemodelan Prediksi Volatilitas Return Saham Sektor Energi LQ45
**Ilda Nurida — 6032251068 | Pembimbing: Dr. Ir. Ahmad Saikhu, S.Si, MT.**

### 🎯 Tujuan (menjawab RM-3)
Membandingkan performa **XGBoost** (tabular/non-sequential) dan **LSTM** (sequential, lookback=10)
secara **adil** untuk memprediksi volatilitas return saham harian.

### 🔧 Perbaikan metodologis dibanding notebook sebelumnya
1. **Skala identik** — kedua model dievaluasi pada **skala volatilitas asli** (inverse-transform MinMax
   diterapkan ke *kedua* model, termasuk XGBoost yang sebelumnya terlewat).
2. **Sampel uji selaras** — XGBoost dievaluasi hanya pada baris `[lookback:]` agar **titik target persis sama**
   dengan LSTM (LSTM kehilangan `lookback` baris pertama karena windowing).
3. **Metrik diperluas** — selain RMSE, MAE, MAPE, ditambahkan **sMAPE, R², dan QLIKE**
   yang lebih sesuai untuk peramalan volatilitas.
4. **Reprodusibel** — notebook berdiri sendiri (load model `.pkl` & `.keras` yang sudah disimpan).

### 📥 Input
- `dataset_test_{emiten}.csv`, `scaler_params_{emiten}.csv`  (per emiten, sudah ter-scale)
- Model tersimpan: `xgboost_{emiten}.pkl`, `lstm_best_{emiten}.keras`

### 📤 Output (→ Bab 4 Tesis)
- `tabel_perbandingan_xgb_lstm.csv` (Tabel 4.X)
- `gambar_perbandingan_xgb_lstm.png` (Gambar 4.X)
- Kesimpulan model terbaik per emiten & keseluruhan


## ⚙️ CELL 1 — Import & Konfigurasi

In [9]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras

# ============================================================
# KONFIGURASI — sesuaikan ROOT bila lokasi folder berbeda
# ============================================================
ROOT = Path(r'e:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS'
            r'\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026')
if not ROOT.exists():
    ROOT = Path('.').resolve()          # fallback: jalankan dari folder proyek

DIR_FINAL   = ROOT / '03_dataset' / 'final'
DIR_XGB     = ROOT / '04_model' / 'xgboost'
DIR_LSTM    = ROOT / '04_model' / 'lstm'
DIR_TABEL   = ROOT / '06_dokumen_tesis' / 'tabel'
DIR_GAMBAR  = ROOT / '06_dokumen_tesis' / 'gambar' / 'model'
for d in (DIR_TABEL, DIR_GAMBAR):
    d.mkdir(parents=True, exist_ok=True)

URUTAN_EMITEN = ['ADRO', 'ADMR', 'ITMG', 'PTBA', 'MEDC', 'AKRA', 'PGAS']
KOLOM_Y       = 'Volatilitas'
LOOKBACK      = 10                       # samakan dgn training LSTM
EPS           = 1e-8

# Daftar fitur X (21). Biarkan None untuk auto-derive dari CSV (semua kolom
# kecuali kolom non-fitur). Jika urutan fitur saat training berbeda, ISI manual
# dengan KOLOM_X_KANDIDAT asli Anda agar urutan fitur identik dgn saat training.
KOLOM_X_MANUAL = [
    # Teknikal (20)
    'Close','Log_Return','MA5','MA10','MA20','EMA12','EMA26',
    'RSI14','MACD','MACD_Signal','MACD_Hist',
    'BB_Upper','BB_Lower','BB_Width','BB_PctB',
    'Stoch_K','Stoch_D','Volume','Volume_Change','Volume_MA5',
    # Sentimen (8)
    'Skor_Sentimen','N_Berita','Ada_Berita',
    'Skor_Sentimen_Lag1','Skor_Sentimen_Lag2',
    'Sentiment_MA3','Sentiment_MA20','Sentiment_STD20',
]
KOLOM_NON_FITUR = {'Date', 'Emiten', KOLOM_Y, 'No', 'Unnamed: 0'}

print('='*65)
print('  ⚖️  EVALUASI PERBANDINGAN — SIAP DIJALANKAN')
print('='*65)
print(f'  ROOT        : {ROOT}')
print(f'  TensorFlow  : {tf.__version__}')
print(f'  Emiten      : {URUTAN_EMITEN}')
print(f'  Lookback    : {LOOKBACK}  | Target Y : {KOLOM_Y}')


  ⚖️  EVALUASI PERBANDINGAN — SIAP DIJALANKAN
  ROOT        : e:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026
  TensorFlow  : 2.21.0
  Emiten      : ['ADRO', 'ADMR', 'ITMG', 'PTBA', 'MEDC', 'AKRA', 'PGAS']
  Lookback    : 10  | Target Y : Volatilitas


## 🔧 CELL 2 — Fungsi Bantu: Metrik, Inverse-Transform, Sekuens

In [10]:
def inverse_transform(nilai, scaler_df, kolom=KOLOM_Y):
    '''Kembalikan nilai ter-MinMax-scale ke skala asli. Identik dgn notebook LSTM.'''
    if scaler_df is None or kolom not in scaler_df.index:
        return np.asarray(nilai, dtype=float)
    cmin = float(scaler_df.loc[kolom, 'min'])
    cmax = float(scaler_df.loc[kolom, 'max'])
    return np.asarray(nilai, dtype=float) * (cmax - cmin) + cmin

def buat_sekuens(X, y, lookback=LOOKBACK):
    '''Windowing untuk LSTM. Output y selaras dgn y[lookback:].'''
    Xs, ys = [], []
    for t in range(lookback, len(X)):
        Xs.append(X[t-lookback:t, :]); ys.append(y[t])
    return np.array(Xs), np.array(ys)

# ---------- Metrik (skala asli) ----------
def hitung_mape(y_true, y_pred, eps=EPS):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    mask = np.abs(y_true) > eps
    return np.nan if mask.sum() == 0 else \
        float(np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100)

def hitung_smape(y_true, y_pred, eps=EPS):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    den = np.abs(y_true) + np.abs(y_pred); mask = den > eps
    return np.nan if mask.sum() == 0 else \
        float(np.mean(2*np.abs(y_pred[mask]-y_true[mask])/den[mask])*100)

def hitung_qlike(y_true, y_pred, eps=1e-12):
    '''QLIKE (Patton, 2011) pada proksi varians h = vol^2. Makin kecil makin baik.'''
    h_t = np.maximum(np.asarray(y_true, float)**2, eps)
    h_p = np.maximum(np.asarray(y_pred, float)**2, eps)
    return float(np.mean(np.log(h_p) + h_t/h_p))

def evaluasi_semua_metrik(y_true, y_pred):
    return {
        'RMSE' : float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE'  : float(mean_absolute_error(y_true, y_pred)),
        'MAPE (%)'  : round(hitung_mape(y_true, y_pred), 2),
        'sMAPE (%)' : round(hitung_smape(y_true, y_pred), 2),
        'R2'   : round(float(r2_score(y_true, y_pred)), 4),
        'QLIKE': round(hitung_qlike(y_true, y_pred), 4),
    }

print('✅ Fungsi bantu siap.')


✅ Fungsi bantu siap.


## 📂 CELL 3 — Load Data Uji, Scaler, dan Daftar Fitur per Emiten

In [11]:
dict_train, dict_test, dict_scaler, dict_kolom_x = {}, {}, {}, {}

for emiten in URUTAN_EMITEN:
    path_tr = DIR_FINAL / f'dataset_train_{emiten}.csv'
    path_te = DIR_FINAL / f'dataset_test_{emiten}.csv'
    path_sc = DIR_FINAL / f'scaler_params_{emiten}.csv'
    if not path_te.exists():
        print(f'  \u26a0\ufe0f  {emiten}: dataset_test tidak ditemukan, skip'); continue

    df_te = pd.read_csv(path_te, encoding='utf-8-sig')
    df_tr = pd.read_csv(path_tr, encoding='utf-8-sig') if path_tr.exists() else None
    for d in (df_tr, df_te):
        if d is not None and 'Date' in d.columns:
            d['Date'] = pd.to_datetime(d['Date'], errors='coerce')

    # \u2500\u2500 Tentukan fitur X (sumber kebenaran = DATA LATIH) \u2500\u2500
    #    Hindari denylist pd data test: test kadang punya kolom ekstra
    #    (mis. OHLCV mentah) yg tak dilihat model \u2192 mismatch jumlah fitur.
    if KOLOM_X_MANUAL is not None:
        sumber = list(KOLOM_X_MANUAL)                                   # allowlist eksplisit
    elif df_tr is not None:
        sumber = [c for c in df_tr.columns if c not in KOLOM_NON_FITUR] # urutan training
    else:
        sumber = [c for c in df_te.columns if c not in KOLOM_NON_FITUR]

    # Hanya pakai kolom yg ADA di train (bila ada) DAN test
    kolom_x = [c for c in sumber
               if c in df_te.columns and (df_tr is None or c in df_tr.columns)]

    df_te = df_te.dropna(subset=kolom_x + [KOLOM_Y]).reset_index(drop=True)
    if df_tr is not None:
        df_tr = df_tr.dropna(subset=kolom_x + [KOLOM_Y]).reset_index(drop=True)

    scaler = pd.read_csv(path_sc, index_col=0, encoding='utf-8-sig') if path_sc.exists() else None

    dict_train[emiten]   = df_tr
    dict_test[emiten]    = df_te
    dict_scaler[emiten]  = scaler
    dict_kolom_x[emiten] = kolom_x

    extra_test = [c for c in df_te.columns
                  if c not in KOLOM_NON_FITUR and c not in kolom_x]
    info_extra = f' | kolom test diabaikan: {len(extra_test)}' if extra_test else ''
    print(f'  \u2705 {emiten}: test={len(df_te):,} | fitur={len(kolom_x)} | '
          f'scaler={"ada" if scaler is not None else "TIDAK ADA"}{info_extra}')

print(f'\n  Total emiten dimuat: {len(dict_test)}')
print('  \u2139\ufe0f  Jika "fitur" \u2260 jumlah fitur saat melatih model, isi KOLOM_X_MANUAL di Cell 1.')


  ✅ ADRO: test=209 | fitur=26 | scaler=ada | kolom test diabaikan: 8
  ✅ ADMR: test=207 | fitur=26 | scaler=ada | kolom test diabaikan: 8
  ✅ ITMG: test=207 | fitur=26 | scaler=ada | kolom test diabaikan: 8
  ✅ PTBA: test=209 | fitur=26 | scaler=ada | kolom test diabaikan: 8
  ✅ MEDC: test=209 | fitur=26 | scaler=ada | kolom test diabaikan: 8
  ✅ AKRA: test=207 | fitur=26 | scaler=ada | kolom test diabaikan: 8
  ✅ PGAS: test=209 | fitur=26 | scaler=ada | kolom test diabaikan: 8

  Total emiten dimuat: 7
  ℹ️  Jika "fitur" ≠ jumlah fitur saat melatih model, isi KOLOM_X_MANUAL di Cell 1.


## 🌳 CELL 4 — Evaluasi XGBoost (skala asli, sampel selaras)
> **Perbaikan kunci:** prediksi di-*inverse-transform* ke skala volatilitas asli, lalu
> dipotong ke baris `[LOOKBACK:]` agar titik ujinya **identik** dengan LSTM.

In [12]:
hasil_xgb = {}

def _scaler_bisa_inverse(scaler_df, kolom=KOLOM_Y):
    """True jika scaler valid utk inverse-transform kolom target (ada index + kolom min/max)."""
    return (scaler_df is not None
            and kolom in scaler_df.index
            and {'min', 'max'}.issubset(set(scaler_df.columns)))

for emiten in URUTAN_EMITEN:
    if emiten not in dict_test:
        continue
    path_mdl = DIR_XGB / f'xgboost_{emiten}.pkl'
    if not path_mdl.exists():
        print(f'  \u26a0\ufe0f  {emiten}: model XGBoost tidak ditemukan, skip'); continue

    model   = joblib.load(path_mdl)
    df_te   = dict_test[emiten]
    kolom_x = list(dict_kolom_x[emiten])
    scaler  = dict_scaler[emiten]

    # \u2500\u2500 Samakan urutan fitur dgn saat training bila model menyimpan nama fitur
    if hasattr(model, 'feature_names_in_'):
        nama_model = list(model.feature_names_in_)
        if set(nama_model).issubset(set(df_te.columns)):
            kolom_x = nama_model
        else:
            print(f'  \u26a0\ufe0f  {emiten}: nama fitur model tidak cocok dgn kolom data.')

    # \u2500\u2500 Validasi jumlah fitur \u2014 SKIP (bukan crash) bila tak cocok
    n_exp = getattr(model, 'n_features_in_', None)
    if n_exp is not None and n_exp != len(kolom_x):
        print(f'  \u274c {emiten}: jumlah fitur tidak cocok \u2014 model butuh {n_exp}, '
              f'tersedia {len(kolom_x)}. Emiten di-SKIP.\n'
              f'      \u2192 Solusi: set KOLOM_X_MANUAL (Cell 1) dgn {n_exp} nama fitur '
              f'persis spt saat melatih model, lalu jalankan ulang Cell 3 & 4.')
        continue

    # \u2500\u2500 Peringatan keras bila scaler tak bisa inverse-transform
    if not _scaler_bisa_inverse(scaler):
        print(f'  \u2757 {emiten}: scaler TIDAK valid utk "{KOLOM_Y}" (min/max tak ditemukan). '
              f'Metrik akan pada SKALA SCALED, tidak sebanding dgn LSTM \u2014 cek scaler_params_{emiten}.csv.')

    X_te        = df_te[kolom_x].values
    y_te_scaled = df_te[KOLOM_Y].values
    try:
        y_pred_scaled = model.predict(X_te)
    except Exception as e:
        print(f'  \u274c {emiten}: gagal prediksi ({type(e).__name__}: {e}). Emiten di-SKIP.')
        continue

    # \u2500\u2500 Selaraskan dgn LSTM: ambil baris [LOOKBACK:]
    if len(y_te_scaled) <= LOOKBACK:
        print(f'  \u26a0\ufe0f  {emiten}: data uji ({len(y_te_scaled)}) \u2264 lookback ({LOOKBACK}), skip')
        continue
    y_te_scaled   = y_te_scaled[LOOKBACK:]
    y_pred_scaled = y_pred_scaled[LOOKBACK:]
    dates = (df_te['Date'].iloc[LOOKBACK:].reset_index(drop=True)
             if 'Date' in df_te.columns else None)

    # \u2500\u2500 Inverse-transform ke skala volatilitas asli
    y_true = inverse_transform(y_te_scaled,   scaler)
    y_pred = np.clip(inverse_transform(y_pred_scaled, scaler), 0, None)

    # \u2500\u2500 Buang pasangan NaN/inf (dan jaga keselarasan dates)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    if valid.sum() == 0:
        print(f'  \u26a0\ufe0f  {emiten}: seluruh nilai NaN/inf, skip'); continue
    y_true, y_pred = y_true[valid], y_pred[valid]
    if dates is not None:
        dates = dates[valid].reset_index(drop=True)

    metrik = evaluasi_semua_metrik(y_true, y_pred)
    hasil_xgb[emiten] = {'y_true': y_true, 'y_pred': y_pred, 'dates': dates, **metrik}
    print(f"  \u2705 {emiten}: RMSE={metrik['RMSE']:.5f} | MAE={metrik['MAE']:.5f} | "
          f"MAPE={metrik['MAPE (%)']:.2f}% | R2={metrik['R2']}")

print(f'\n  XGBoost dievaluasi: {len(hasil_xgb)} emiten')


  ❌ ADRO: jumlah fitur tidak cocok — model butuh 27, tersedia 26. Emiten di-SKIP.
      → Solusi: set KOLOM_X_MANUAL (Cell 1) dgn 27 nama fitur persis spt saat melatih model, lalu jalankan ulang Cell 3 & 4.
  ❌ ADMR: jumlah fitur tidak cocok — model butuh 27, tersedia 26. Emiten di-SKIP.
      → Solusi: set KOLOM_X_MANUAL (Cell 1) dgn 27 nama fitur persis spt saat melatih model, lalu jalankan ulang Cell 3 & 4.
  ❌ ITMG: jumlah fitur tidak cocok — model butuh 27, tersedia 26. Emiten di-SKIP.
      → Solusi: set KOLOM_X_MANUAL (Cell 1) dgn 27 nama fitur persis spt saat melatih model, lalu jalankan ulang Cell 3 & 4.
  ❌ PTBA: jumlah fitur tidak cocok — model butuh 27, tersedia 26. Emiten di-SKIP.
      → Solusi: set KOLOM_X_MANUAL (Cell 1) dgn 27 nama fitur persis spt saat melatih model, lalu jalankan ulang Cell 3 & 4.
  ❌ MEDC: jumlah fitur tidak cocok — model butuh 27, tersedia 26. Emiten di-SKIP.
      → Solusi: set KOLOM_X_MANUAL (Cell 1) dgn 27 nama fitur persis spt saat melatih mode

## 🧠 CELL 5 — Evaluasi LSTM (skala asli)

In [13]:
hasil_lstm = {}

for emiten in URUTAN_EMITEN:
    if emiten not in dict_test:
        continue
    path_mdl = DIR_LSTM / f'lstm_best_{emiten}.keras'
    if not path_mdl.exists():
        print(f'  ⚠️  {emiten}: model LSTM tidak ditemukan, skip'); continue

    model   = keras.models.load_model(path_mdl, compile=False)
    df_te   = dict_test[emiten]
    kolom_x = dict_kolom_x[emiten]
    scaler  = dict_scaler[emiten]

    X_te = df_te[kolom_x].values.astype(float)
    y_te = df_te[KOLOM_Y].values.astype(float)
    X_seq, y_seq = buat_sekuens(X_te, y_te, LOOKBACK)

    # Validasi bentuk input
    n_exp = model.input_shape[-1]
    if n_exp != X_seq.shape[-1]:
        print(f'  ❌ {emiten}: fitur LSTM ({X_seq.shape[-1]}) != input model ({n_exp}). '
              f'Emiten di-SKIP — set KOLOM_X_MANUAL agar konsisten.'); continue

    y_pred_scaled = model.predict(X_seq, verbose=0).flatten()
    dates = df_te['Date'].iloc[LOOKBACK:].reset_index(drop=True) if 'Date' in df_te.columns else None

    y_true = inverse_transform(y_seq,        scaler)
    y_pred = np.clip(inverse_transform(y_pred_scaled, scaler), 0, None)

    metrik = evaluasi_semua_metrik(y_true, y_pred)
    hasil_lstm[emiten] = {'y_true': y_true, 'y_pred': y_pred, 'dates': dates, **metrik}
    print(f"  ✅ {emiten}: RMSE={metrik['RMSE']:.5f} | MAE={metrik['MAE']:.5f} | "
          f"MAPE={metrik['MAPE (%)']:.2f}% | R2={metrik['R2']}")

print(f'\n  LSTM dievaluasi: {len(hasil_lstm)} emiten')


  ❌ ADRO: fitur LSTM (26) != input model (28). Emiten di-SKIP — set KOLOM_X_MANUAL agar konsisten.
  ❌ ADMR: fitur LSTM (26) != input model (28). Emiten di-SKIP — set KOLOM_X_MANUAL agar konsisten.
  ❌ ITMG: fitur LSTM (26) != input model (28). Emiten di-SKIP — set KOLOM_X_MANUAL agar konsisten.
  ❌ PTBA: fitur LSTM (26) != input model (28). Emiten di-SKIP — set KOLOM_X_MANUAL agar konsisten.
  ❌ MEDC: fitur LSTM (26) != input model (28). Emiten di-SKIP — set KOLOM_X_MANUAL agar konsisten.
  ❌ AKRA: fitur LSTM (26) != input model (28). Emiten di-SKIP — set KOLOM_X_MANUAL agar konsisten.
  ❌ PGAS: fitur LSTM (26) != input model (28). Emiten di-SKIP — set KOLOM_X_MANUAL agar konsisten.

  LSTM dievaluasi: 0 emiten


## 📋 CELL 6 — Tabel Perbandingan Terpadu (→ Tabel 4.X)
Kolom **Model Terbaik** ditentukan berdasarkan **RMSE** terkecil (skala asli, sampel identik).

In [14]:
baris = []
emiten_eval = [e for e in URUTAN_EMITEN if e in hasil_xgb and e in hasil_lstm]

for e in emiten_eval:
    x, l = hasil_xgb[e], hasil_lstm[e]
    menang = 'LSTM' if l['RMSE'] < x['RMSE'] else 'XGBoost'
    baris.append({
        'Emiten'      : e,
        'RMSE_XGB'    : round(x['RMSE'], 6),  'RMSE_LSTM'    : round(l['RMSE'], 6),
        'MAE_XGB'     : round(x['MAE'], 6),   'MAE_LSTM'     : round(l['MAE'], 6),
        'MAPE_XGB(%)' : x['MAPE (%)'],        'MAPE_LSTM(%)' : l['MAPE (%)'],
        'R2_XGB'      : x['R2'],              'R2_LSTM'      : l['R2'],
        'QLIKE_XGB'   : x['QLIKE'],           'QLIKE_LSTM'   : l['QLIKE'],
        'Model Terbaik': menang,
    })

df_cmp = pd.DataFrame(baris)

# Baris rata-rata
if len(df_cmp):
    avg = {'Emiten': 'RATA-RATA'}
    for c in df_cmp.columns:
        if c not in ('Emiten', 'Model Terbaik'):
            avg[c] = round(df_cmp[c].mean(), 6)
    menang_avg = 'LSTM' if avg['RMSE_LSTM'] < avg['RMSE_XGB'] else 'XGBoost'
    avg['Model Terbaik'] = menang_avg
    df_cmp = pd.concat([df_cmp, pd.DataFrame([avg])], ignore_index=True)

df_cmp.index = list(range(1, len(df_cmp))) + ['—']
df_cmp.index.name = 'No'

pd.set_option('display.width', 200, 'display.max_columns', None)
print('='*100)
print('  📋 TABEL PERBANDINGAN XGBoost vs LSTM — skala volatilitas asli')
print('='*100)
print(df_cmp.to_string())

path_cmp = DIR_TABEL / 'tabel_perbandingan_xgb_lstm.csv'
df_cmp.to_csv(path_cmp, encoding='utf-8-sig')
print(f'\n  💾 Tabel disimpan: {path_cmp}')


  📋 TABEL PERBANDINGAN XGBoost vs LSTM — skala volatilitas asli
Empty DataFrame
Columns: []
Index: [—]

  💾 Tabel disimpan: e:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026\06_dokumen_tesis\tabel\tabel_perbandingan_xgb_lstm.csv


## 📊 CELL 7 — Grafik Perbandingan RMSE & MAPE per Emiten (→ Gambar 4.X)

In [15]:
if len(emiten_eval):
    x_pos = np.arange(len(emiten_eval)); w = 0.38
    fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))

    # RMSE
    ax[0].bar(x_pos - w/2, [hasil_xgb[e]['RMSE']  for e in emiten_eval], w, label='XGBoost')
    ax[0].bar(x_pos + w/2, [hasil_lstm[e]['RMSE'] for e in emiten_eval], w, label='LSTM')
    ax[0].set_title('RMSE per Emiten (skala asli — makin kecil makin baik)', fontweight='bold')
    ax[0].set_xticks(x_pos); ax[0].set_xticklabels(emiten_eval); ax[0].set_ylabel('RMSE')
    ax[0].legend(); ax[0].grid(axis='y', alpha=0.3)

    # MAPE
    ax[1].bar(x_pos - w/2, [hasil_xgb[e]['MAPE (%)']  for e in emiten_eval], w, label='XGBoost')
    ax[1].bar(x_pos + w/2, [hasil_lstm[e]['MAPE (%)'] for e in emiten_eval], w, label='LSTM')
    ax[1].axhline(20, ls='--', lw=1, color='red', label='Target 20%')
    ax[1].set_title('MAPE per Emiten (%)', fontweight='bold')
    ax[1].set_xticks(x_pos); ax[1].set_xticklabels(emiten_eval); ax[1].set_ylabel('MAPE (%)')
    ax[1].legend(); ax[1].grid(axis='y', alpha=0.3)

    plt.suptitle('Perbandingan Performa XGBoost vs LSTM — Prediksi Volatilitas Sektor Energi LQ45',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    path_fig = DIR_GAMBAR / 'gambar_perbandingan_xgb_lstm.png'
    plt.savefig(path_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  💾 Gambar disimpan: {path_fig}')
else:
    print('  ⚠️  Tidak ada emiten untuk divisualisasikan.')


  ⚠️  Tidak ada emiten untuk divisualisasikan.


## 🏁 CELL 8 — Kesimpulan (menjawab RM-3)

In [16]:
if len(df_cmp) > 1:
    df_emiten = df_cmp[df_cmp['Emiten'] != 'RATA-RATA']
    n_lstm = (df_emiten['Model Terbaik'] == 'LSTM').sum()
    n_xgb  = (df_emiten['Model Terbaik'] == 'XGBoost').sum()
    avg_row = df_cmp[df_cmp['Emiten'] == 'RATA-RATA'].iloc[0]
    pemenang = avg_row['Model Terbaik']

    print('='*65)
    print('  🏁 KESIMPULAN PERBANDINGAN (RM-3)')
    print('='*65)
    print(f"  Menang per emiten (RMSE) : LSTM {n_lstm}/{len(df_emiten)} | "
          f"XGBoost {n_xgb}/{len(df_emiten)}")
    print(f"  Rata-rata RMSE  : XGBoost={avg_row['RMSE_XGB']:.5f} | LSTM={avg_row['RMSE_LSTM']:.5f}")
    print(f"  Rata-rata MAPE  : XGBoost={avg_row['MAPE_XGB(%)']:.2f}% | LSTM={avg_row['MAPE_LSTM(%)']:.2f}%")
    print(f"  Rata-rata R2    : XGBoost={avg_row['R2_XGB']:.4f} | LSTM={avg_row['R2_LSTM']:.4f}")
    print()
    print(f"  ➤ Model terbaik secara keseluruhan: **{pemenang}**")
    print('='*65)
else:
    print('  ⚠️  Data perbandingan belum lengkap — pastikan kedua model & data uji tersedia.')


  ⚠️  Data perbandingan belum lengkap — pastikan kedua model & data uji tersedia.
